# What estimator does CausalPFN learn? Full-scale run

**Before running:** Settings → Accelerator → **GPU T4 x2** (the code uses one of the two GPUs). Internet can stay **off** unless `TABPFN=True`.

Set `PHASE` below, then **Save Version → Save & Run All (Commit)**. Run the phases in order: `checks` → `h1` → `h2` → `h3`.
For each later run, add this notebook's previous version output as an input (Add Input → Your Work → this notebook); it is picked up automatically for resuming.

In [ ]:
PHASE  = "checks"     # "checks" | "h1" | "h2" | "h3"
HOURS  = 11.3         # stop cleanly before Kaggle's 12 h limit; rerun to resume
SUITES = "easy,hard,clip"
R      = 20
TABPFN = False        # True needs Internet ON (adds TabPFN-as-S-learner to H1)

In [ ]:
import glob, os, subprocess, sys, zipfile

def find_bundle():
    # 1) extracted dataset anywhere under /kaggle/input (follow symlinks)
    for root, dirs, files in os.walk('/kaggle/input', followlinks=True):
        if 'run_kaggle.py' in files and 'causalpfn_v0.pt' in files:
            return root
    # 2) dataset kept as a zip -> extract to /kaggle/working/bundle
    for root, dirs, files in os.walk('/kaggle/input', followlinks=True):
        for f in files:
            if f.endswith('.zip'):
                z = os.path.join(root, f)
                try:
                    names = zipfile.ZipFile(z).namelist()
                except zipfile.BadZipFile:
                    continue
                if any(n.endswith('run_kaggle.py') for n in names):
                    zipfile.ZipFile(z).extractall('/kaggle/working/bundle')
                    for r2, _, f2 in os.walk('/kaggle/working/bundle'):
                        if 'run_kaggle.py' in f2: return r2
    return None

BUNDLE = find_bundle()
if BUNDLE is None:
    print('Bundle NOT found. Contents of /kaggle/input (3 levels):')
    for root, dirs, files in os.walk('/kaggle/input', followlinks=True):
        depth = root.count(os.sep) - 2
        if depth > 3: continue
        print('  ' * depth + os.path.basename(root) + '/', files[:8])
    raise SystemExit('Attach the cfm-bundle dataset: right panel -> Add Input -> Datasets -> Your Datasets -> cfm-bundle')
print('bundle:', BUNDLE, sorted(os.listdir(BUNDLE)))
assert os.path.exists(os.path.join(BUNDLE, 'causalpfn_v0.pt')), 'checkpoint missing from bundle'
prev = [d for d in glob.glob('/kaggle/input/**/results', recursive=True) if os.path.isdir(d)]
print('previous results found for resume:', prev)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
if TABPFN:
    !pip -q install tabpfn
cmd = [sys.executable, os.path.join(BUNDLE, 'run_kaggle.py'), '--bundle', BUNDLE, '--phase', PHASE,
       '--hours', str(HOURS), '--suites', SUITES, '--R', str(R), '--n_jobs', '4', '--out', '/kaggle/working/results']
for p in prev: cmd += ['--resume_from', p]
if TABPFN: cmd += ['--tabpfn']
print(' '.join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout: print(line, end='')
proc.wait(); print('exit code', proc.returncode)

In [ ]:
# summary tables from everything produced so far
!python {BUNDLE}/analyze_full.py /kaggle/working/results | tail -80